# Module 10 Instructor Colab Notebook: Conditional Probability and Bayesian Inference

## A Mathematical Bridge to Artificial Intelligence

Conditioning means restricting attention to cases where some information is already known. This simple change of reference population connects joint and marginal probabilities, Bayes' theorem, probabilistic classification, and generative AI.

A language model predicts the next token conditional on earlier tokens. A Naive Bayes classifier predicts a class conditional on observed features. Bayesian inference reverses a conditional relationship by combining prior beliefs with evidence. This notebook builds each idea from probability tables before moving to AI applications.

**Recommended pacing:** three class meetings. The conditional-independence examples deserve time because they prevent a common misconception: ordinary independence and conditional independence are different statements.


> **How to use this notebook in class**
>
> For every conditional probability, ask students to name the restricted reference group and the event counted inside it. Use tables before formulas, and require students to identify priors, likelihoods, evidence, and posteriors in words before substituting numbers. In Naive Bayes, distinguish the mathematical assumption from the fact that the features may still be marginally associated.

## Notebook map

| Section | Main mathematical ideas | Direct AI connection |
|---|---|---|
| 1 | Joint, marginal, and conditional probability | Feature-label tables |
| 2 | Total probability and Bayes' theorem | Reversing evidence and class conditionals |
| 3 | Independence and conditional independence | Graphical-model assumptions |
| 4 | Conditional sequence models | Next-token generative probabilities |
| 5 | Bayesian inference and updating | Combining prior information with data |
| 6 | Naive Bayes text classification | Spam and document classification |
| Integrated lab | Smoothed Naive Bayes | Train and classify short documents |


In [ ]:
# Core setup: compatible with Google Colab and standard Jupyter.
import math
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

np.set_printoptions(precision=5, suppress=True)
plt.rcParams.update({
    "figure.figsize": (8, 5),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

try:
    import ipywidgets as widgets
    from ipywidgets import interact, FloatSlider, IntSlider
    WIDGETS_AVAILABLE = True
except Exception:
    WIDGETS_AVAILABLE = False

print("Setup complete.")
print("Interactive widgets available:", WIDGETS_AVAILABLE)


In [ ]:
# Reusable instructor-solution helper.
def reveal_markdown(button_text, solution_markdown):
    if not WIDGETS_AVAILABLE:
        display(Markdown("**Instructor solution**"))
        display(Markdown(solution_markdown))
        return

    button = widgets.Button(description=button_text, button_style="info")
    output = widgets.Output()
    state = {"open": False}

    def toggle(_):
        with output:
            output.clear_output()
            state["open"] = not state["open"]
            if state["open"]:
                display(Markdown(solution_markdown))
                button.description = "Hide instructor solution"
            else:
                button.description = button_text

    button.on_click(toggle)
    display(button, output)


## Module learning objectives

By the end of the module, students should be able to:

1. compute joint, marginal, and conditional probabilities from a table;
2. apply the law of total probability;
3. derive and use Bayes' theorem;
4. distinguish independence from conditional independence;
5. factor a sequence probability using the probability chain rule;
6. identify prior, likelihood, evidence, and posterior;
7. apply the Naive Bayes assumption; and
8. use Laplace smoothing and log scores in text classification.

## Prerequisite checkpoint

Students should know events, random variables, PMFs, distributions, sums, products, logarithms, and basic set intersections.


---

# Section 1: Joint, Marginal, and Conditional Probability

## 1. Joint probability

The **joint probability** of events $$A$$ and $$B$$ is

$$
P(A\cap B),
$$

the probability that both occur.

For random variables $$X$$ and $$Y,$$ a joint PMF is

$$
p_{X,Y}(x,y)=P(X=x,Y=y).
$$

It must be nonnegative and sum to one over all pairs:

$$
\sum_x\sum_y p_{X,Y}(x,y)=1.
$$

A joint distribution contains information about each variable and about how they occur together.


## 2. A complete two-way table

Suppose one hundred AI service periods are classified by weather and traffic load:

| | High load | Low load | Row total |
|---|---:|---:|---:|
| Rain | 18 | 12 | 30 |
| No rain | 14 | 56 | 70 |
| Column total | 32 | 68 | 100 |

The joint probability of rain and high load is

$$
P(R\cap H)=\frac{18}{100}=0.18.
$$

The joint probability of no rain and low load is

$$
P(R^c\cap H^c)=\frac{56}{100}=0.56.
$$

Every interior cell is a joint count, and dividing by the grand total gives a joint probability.


In [ ]:
joint_counts10 = np.array([[18, 12], [14, 56]])
joint_probabilities10 = joint_counts10/joint_counts10.sum()

fig, ax = plt.subplots(figsize=(6.5, 4.8))
image_joint10 = ax.imshow(joint_probabilities10, cmap="Blues", vmin=0, vmax=0.6)
ax.set_xticks([0, 1], ["high load", "low load"])
ax.set_yticks([0, 1], ["rain", "no rain"])
for i in range(2):
    for j in range(2):
        ax.text(j, i, f"count = {joint_counts10[i,j]}\nprobability = {joint_probabilities10[i,j]:.2f}",
                ha="center", va="center",
                color="white" if joint_probabilities10[i,j] > 0.3 else "black")
ax.set_title("Joint probabilities occupy table cells")
fig.colorbar(image_joint10, ax=ax, label="joint probability")
plt.tight_layout()
plt.show()


## 3. Marginal probability

A **marginal probability** describes one variable without specifying the other. It is obtained by summing the joint distribution across the unwanted variable.

From the table,

$$
P(R)=P(R\cap H)+P(R\cap H^c).
$$

Thus

$$
P(R)=0.18+0.12=0.30.
$$

Similarly,

$$
P(H)=0.18+0.14=0.32.
$$

The word “marginal” comes from row and column totals traditionally written in the margins of a table.


## 4. Conditional probability

The **conditional probability** of $$A$$ given $$B$$ is

$$
\boxed{P(A\mid B)=\frac{P(A\cap B)}{P(B)},
\qquad P(B)>0.}
$$

Conditioning on $$B$$ restricts the reference population to cases where $$B$$ occurred. The numerator counts cases satisfying both $$A$$ and $$B,$$ while the denominator counts all cases satisfying $$B.$$

Using the table,

$$
P(H\mid R)=\frac{P(H\cap R)}{P(R)}
=\frac{0.18}{0.30}
=\boxed{0.60}.
$$

Among rainy periods, sixty percent have high load.

In the reverse direction,

$$
P(R\mid H)=\frac{0.18}{0.32}
=\boxed{0.5625}.
$$

These conditionals are different because their reference populations are different.


## 5. Multiplication rule

Rearrange the conditional-probability definition:

$$
P(A\cap B)=P(A\mid B)P(B).
$$

Equally,

$$
P(A\cap B)=P(B\mid A)P(A).
$$

Thus

$$
\boxed{P(A\cap B)=P(A\mid B)P(B)=P(B\mid A)P(A).}
$$

The multiplication rule converts a conditional probability and a marginal probability into a joint probability.


## 6. Joint, marginal, and conditional distributions

For discrete random variables:

$$
p_X(x)=\sum_y p_{X,Y}(x,y)
$$

is a marginal PMF, and

$$
p_{X\mid Y}(x\mid y)
=\frac{p_{X,Y}(x,y)}{p_Y(y)}
$$

is a conditional PMF when $$p_Y(y)>0.$$

For continuous variables, sums become integrals and PMFs become densities. The conceptual relationships remain the same.


> **Common mistakes in Section 1**
>
> - Reversing the event after the conditioning bar.
> - Dividing by the wrong marginal probability.
> - Assuming $$P(A\mid B)=P(B\mid A).$$
> - Adding joint cells outside the restricted reference group.
> - Confusing a marginal probability with a joint table cell.

## Section 1 terminology

| Term | Meaning |
|---|---|
| Joint probability | Probability that specified events occur together |
| Joint distribution | Probability assignment over combinations of variable values |
| Marginal probability | Probability of one event or variable after summing over others |
| Conditional probability | Probability inside a restricted reference group |
| Conditioning event | Event to the right of the conditioning bar |
| Multiplication rule | Joint probability as conditional times marginal probability |
| Two-way table | Table displaying joint counts or probabilities for two variables |


## Section 1 guided worksheet

Use the weather-load table.

1. Compute $$P(H^c).$$
2. Compute $$P(R^c\cap H).$$
3. Compute $$P(H\mid R^c).$$
4. Compute $$P(R^c\mid H).$$
5. Explain why the last two values should not be expected to agree.


In [ ]:
reveal_markdown("Open Section 1 solutions", r"""
1. Low load occurs in sixty-eight of one hundred periods:

   $$
   P(H^c)=\frac{68}{100}=\boxed{0.68}.
   $$

2. No rain and high load occurs fourteen times:

   $$
   P(R^c\cap H)=\frac{14}{100}=\boxed{0.14}.
   $$

3. Among seventy no-rain periods, fourteen have high load:

   $$
   P(H\mid R^c)=\frac{14}{70}=\boxed{0.20}.
   $$

4. Among thirty-two high-load periods, fourteen have no rain:

   $$
   P(R^c\mid H)=\frac{14}{32}=\boxed{0.4375}.
   $$

5. The first conditions on no rain and uses seventy cases as its reference group. The second conditions on high load and uses thirty-two cases. Different denominators answer different questions.
""")


---

# Section 2: The Law of Total Probability and Bayes' Theorem

## 1. Partitioning the sample space

Events $$B_1,\ldots,B_k$$ form a **partition** when they are mutually exclusive and cover the sample space:

$$
B_i\cap B_j=\varnothing
\quad\text{for }i\ne j,
$$

and

$$
B_1\cup\cdots\cup B_k=\Omega.
$$

Every occurrence of event $$A$$ must occur together with exactly one partition event. Therefore

$$
P(A)=\sum_{i=1}^{k}P(A\cap B_i).
$$

Using the multiplication rule gives the **law of total probability**:

$$
\boxed{P(A)=\sum_{i=1}^{k}P(A\mid B_i)P(B_i).}
$$


## 2. Deriving Bayes' theorem

The multiplication rule gives

$$
P(A\cap B)=P(B\mid A)P(A).
$$

The conditional probability of $$A$$ given $$B$$ is

$$
P(A\mid B)=\frac{P(A\cap B)}{P(B)}.
$$

Substitute the multiplication rule into the numerator:

$$
\boxed{P(A\mid B)=\frac{P(B\mid A)P(A)}{P(B)}.}
$$

If $$A_1,\ldots,A_k$$ form a partition, the denominator can be expanded:

$$
\boxed{P(A_j\mid B)
=\frac{P(B\mid A_j)P(A_j)}
{\sum_{i=1}^{k}P(B\mid A_i)P(A_i)}.}
$$


## 3. The four Bayesian roles

In

$$
P(A\mid B)=\frac{P(B\mid A)P(A)}{P(B)},
$$

the terms have distinct roles:

| Quantity | Name | Interpretation |
|---|---|---|
| $$P(A)$$ | Prior | Belief about $$A$$ before observing $$B$$ |
| $$P(B\mid A)$$ | Likelihood | Compatibility of evidence $$B$$ with $$A$$ |
| $$P(B)$$ | Evidence | Overall probability of observing $$B$$ |
| $$P(A\mid B)$$ | Posterior | Updated belief about $$A$$ after observing $$B$$ |

The evidence normalizes the numerator so posterior probabilities sum to one across competing hypotheses.


## 4. Detailed spam-evidence example

Suppose

$$
P(S)=0.20
$$

for spam, and let $$F$$ be the event that an email contains the word “free.” Assume

$$
P(F\mid S)=0.70
$$

and

$$
P(F\mid S^c)=0.10.
$$

First compute the evidence probability:

$$
P(F)=P(F\mid S)P(S)+P(F\mid S^c)P(S^c).
$$

Substitute:

$$
P(F)=(0.70)(0.20)+(0.10)(0.80).
$$

Therefore

$$
P(F)=0.14+0.08=0.22.
$$

Bayes' theorem gives

$$
P(S\mid F)=\frac{P(F\mid S)P(S)}{P(F)}
=\frac{(0.70)(0.20)}{0.22}.
$$

Thus

$$
\boxed{P(S\mid F)=\frac{0.14}{0.22}\approx 0.636364.}
$$

The word increases the spam probability from twenty percent to about sixty-three point six percent, but it does not make spam certain.


In [ ]:
prior_spam10 = 0.20
likelihood_free_spam10 = 0.70
likelihood_free_ham10 = 0.10
evidence_free10 = likelihood_free_spam10*prior_spam10 + likelihood_free_ham10*(1-prior_spam10)
posterior_spam_free10 = likelihood_free_spam10*prior_spam10/evidence_free10

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar(["prior spam", "posterior after 'free'"],
       [prior_spam10, posterior_spam_free10], color=["gray", "crimson"])
ax.set_ylim(0, 1)
ax.set_ylabel("probability")
ax.set_title("Evidence updates rather than replaces the prior")
for index, value in enumerate([prior_spam10, posterior_spam_free10]):
    ax.text(index, value+0.03, f"{value:.3f}", ha="center")
plt.show()


## 5. Base-rate effects

Even strong evidence can produce a modest posterior when the prior event is rare. The **base rate** is the prior frequency of the event in the relevant population.

Ignoring the base rate replaces

$$
P(A\mid B)
$$

with the likelihood

$$
P(B\mid A),
$$

which answers a different question. This error is called **base-rate neglect**.

In AI evaluation, a detector can have high sensitivity but still produce many false alerts when true anomalies are extremely rare.


> **Common mistakes in Section 2**
>
> - Replacing the posterior with the likelihood.
> - Omitting prior probabilities.
> - Forgetting alternative causes in the evidence denominator.
> - Treating the evidence as an arbitrary scaling constant without interpretation.
> - Ignoring rare-event base rates.
> - Assuming one piece of evidence makes a posterior equal to zero or one.

## Section 2 terminology

| Term | Meaning |
|---|---|
| Partition | Mutually exclusive events covering the sample space |
| Law of total probability | Weighted sum across partition-based conditionals |
| Bayes' theorem | Rule reversing a conditional relationship |
| Prior | Probability before current evidence |
| Likelihood | Probability of evidence under a hypothesis |
| Evidence | Overall probability of observed information |
| Posterior | Probability after incorporating evidence |
| Base rate | Prior event frequency in the relevant population |


## Section 2 guided worksheet

An anomaly occurs in one percent of transactions. A detector alerts on ninety-five percent of anomalies and on four percent of normal transactions.

1. Define events for anomaly and alert.
2. Compute the overall alert probability.
3. Compute the probability of an anomaly given an alert.
4. Explain why this posterior is much smaller than ninety-five percent.


In [ ]:
reveal_markdown("Open Section 2 solutions", r"""
Let $$A$$ be anomaly and $$D$$ be detector alert. Then

$$
P(A)=0.01,
$$

$$
P(D\mid A)=0.95,
$$

and

$$
P(D\mid A^c)=0.04.
$$

The overall alert probability is

$$
P(D)=(0.95)(0.01)+(0.04)(0.99).
$$

Compute:

$$
P(D)=0.0095+0.0396=\boxed{0.0491}.
$$

Bayes' theorem gives

$$
P(A\mid D)=\frac{(0.95)(0.01)}{0.0491}
=\frac{0.0095}{0.0491}
\approx \boxed{0.193483}.
$$

The sensitivity ninety-five percent is $$P(D\mid A),$$ not $$P(A\mid D).$$ Normal transactions are much more common, so even a four-percent false-alert rate contributes many alerts.
""")


---

# Section 3: Independence and Conditional Independence

## 1. Ordinary independence

Events $$A$$ and $$B$$ are **independent** when

$$
\boxed{P(A\cap B)=P(A)P(B).}
$$

When probabilities are positive, equivalent statements are

$$
P(A\mid B)=P(A)
$$

and

$$
P(B\mid A)=P(B).
$$

Independence means knowing one event does not change the probability of the other. It does not mean the events are mutually exclusive. Two nonzero mutually exclusive events are necessarily dependent because they cannot occur together.


## 2. Checking independence in the weather table

From Section 1,

$$
P(R)=0.30,
$$

$$
P(H)=0.32,
$$

and

$$
P(R\cap H)=0.18.
$$

If rain and high load were independent, their joint probability would be

$$
P(R)P(H)=(0.30)(0.32)=0.096.
$$

Because

$$
0.18\ne 0.096,
$$

the events are not independent in this table.

Equivalently,

$$
P(H\mid R)=0.60\ne 0.32=P(H).
$$


## 3. Conditional independence

Events $$A$$ and $$B$$ are **conditionally independent given** $$C$$ when

$$
\boxed{P(A\cap B\mid C)
=P(A\mid C)P(B\mid C).}
$$

We write

$$
A\perp B\mid C.
$$

Conditioning can create independence, remove independence, or change the strength of association. Therefore

$$
A\perp B
$$

and

$$
A\perp B\mid C
$$

are different claims.


## 4. Features independent within class but dependent overall

Let $$C$$ denote class, $$A$$ indicate the word “offer,” and $$B$$ indicate the word “win.” Suppose

$$
P(C=S)=0.4,
\qquad
P(C=H)=0.6.
$$

Within spam, assume

$$
P(A\mid S)=0.8,
\qquad
P(B\mid S)=0.5,
$$

and conditional independence, so

$$
P(A\cap B\mid S)=(0.8)(0.5)=0.4.
$$

Within ham, assume

$$
P(A\mid H)=0.1,
\qquad
P(B\mid H)=0.05,
$$

so

$$
P(A\cap B\mid H)=(0.1)(0.05)=0.005.
$$

Marginally,

$$
P(A)=(0.4)(0.8)+(0.6)(0.1)=0.38,
$$

$$
P(B)=(0.4)(0.5)+(0.6)(0.05)=0.23,
$$

and

$$
P(A\cap B)=(0.4)(0.4)+(0.6)(0.005)=0.163.
$$

But

$$
P(A)P(B)=(0.38)(0.23)=0.0874.
$$

Thus the words are conditionally independent within each class by assumption, yet dependent in the overall mixture.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.axis("off")
positions_ci10 = {"class": (0.18, 0.5), "offer": (0.70, 0.72), "win": (0.70, 0.28)}
labels_ci10 = {"class": "email class", "offer": "word: offer", "win": "word: win"}
for key, position in positions_ci10.items():
    ax.text(position[0], position[1], labels_ci10[key], ha="center", va="center",
            bbox=dict(boxstyle="round,pad=0.55", facecolor="#dcfce7", edgecolor="#15803d"))
for target in ["offer", "win"]:
    ax.annotate("", xy=positions_ci10[target], xytext=positions_ci10["class"],
                arrowprops=dict(arrowstyle="->", linewidth=2, shrinkA=35, shrinkB=35))
ax.set_title("A shared class can explain feature association")
plt.show()


## 5. Why conditional independence matters in AI

Conditional independence allows a complex joint distribution to factor into smaller pieces. If features $$X_1,\ldots,X_d$$ are conditionally independent given class $$C,$$ then

$$
P(X_1=x_1,\ldots,X_d=x_d\mid C=c)
=\prod_{j=1}^{d}P(X_j=x_j\mid C=c).
$$

This factorization reduces the number of quantities that must be estimated. The assumption may be imperfect, but the resulting classifier can still be useful.

Graphical models encode such assumptions with nodes and directed or undirected connections. Missing connections have precise meanings only under the chosen graph semantics.


> **Common mistakes in Section 3**
>
> - Confusing independence with mutual exclusivity.
> - Checking only whether two probabilities look numerically close.
> - Assuming marginal independence implies conditional independence.
> - Assuming conditional independence implies marginal independence.
> - Treating a modeling assumption as a proven property of data.
> - Interpreting every graph arrow as direct physical causation.

## Section 3 terminology

| Term | Meaning |
|---|---|
| Independence | Joint probability factors into marginal probabilities |
| Conditional independence | Joint conditional factors after fixing another variable |
| Marginal dependence | Association before conditioning |
| Mutual exclusivity | Events cannot occur together |
| Factorization | Rewriting a joint distribution as products of simpler terms |
| Graphical model | Graph-based representation of probabilistic dependencies |
| Class-conditional model | Feature distribution specified separately within each class |


## Section 3 guided worksheet

Suppose

$$
P(A)=0.5,
\qquad
P(B)=0.4,
\qquad
P(A\cap B)=0.2.
$$

1. Determine whether $$A$$ and $$B$$ are independent.
2. Compute $$P(A\mid B).$$
3. Explain why independent events with positive probability cannot be mutually exclusive.
4. State in words what $$X_1\perp X_2\mid C$$ means in a classifier.


In [ ]:
reveal_markdown("Open Section 3 solutions", r"""
1. Compute the product:

   $$
   P(A)P(B)=(0.5)(0.4)=0.2.
   $$

   Since this equals $$P(A\cap B),$$ the events are independent.

2. We have

   $$
   P(A\mid B)=\frac{P(A\cap B)}{P(B)}
   =\frac{0.2}{0.4}
   =\boxed{0.5}.
   $$

   This equals $$P(A),$$ confirming independence.

3. Mutually exclusive events have $$P(A\cap B)=0.$$ Independent positive-probability events have $$P(A\cap B)=P(A)P(B)>0.$$ Both cannot hold simultaneously.

4. Once class $$C$$ is fixed, knowing feature $$X_1$$ does not change the conditional distribution of feature $$X_2.$$ Their joint class-conditional distribution factors into a product.
""")


---

# Section 4: Conditional Sequence Models and Generative AI

## 1. Probability chain rule

The multiplication rule extends to several variables:

$$
P(X_1=x_1,\ldots,X_T=x_T)
=P(X_1=x_1)
\prod_{t=2}^{T}P(X_t=x_t\mid X_1=x_1,\ldots,X_{t-1}=x_{t-1}).
$$

Using compact notation,

$$
\boxed{P(x_1,\ldots,x_T)
=\prod_{t=1}^{T}P(x_t\mid x_{<t}),}
$$

where $$x_{<t}$$ denotes all earlier values. This identity does not assume independence.


## 2. Tokens, vocabulary, and context

A **token** is one unit processed by a language model. It may represent a word, word piece, punctuation mark, or other encoded unit. The **vocabulary** is the finite set of possible tokens.

The **context** at step $$t$$ is the preceding token sequence

$$
x_{<t}=(x_1,\ldots,x_{t-1}).
$$

An autoregressive language model estimates a conditional distribution

$$
P(X_t=\text{token}\mid x_{<t})
$$

over the vocabulary. A generated token is selected according to a decoding rule based on that distribution, then appended to the context.


## 3. Detailed sequence probability

Consider the three-token sequence

$$
\text{AI learns quickly}.
$$

Suppose a simple model assigns

$$
P(\text{AI})=0.20,
$$

$$
P(\text{learns}\mid\text{AI})=0.50,
$$

and

$$
P(\text{quickly}\mid\text{AI learns})=0.40.
$$

The sequence probability is

$$
P(\text{AI learns quickly})
=(0.20)(0.50)(0.40).
$$

Therefore

$$
\boxed{P(\text{AI learns quickly})=0.04.}
$$

The low joint probability of one complete sequence is not surprising because probability is distributed over many possible sequences.


## 4. Log-probability of a sequence

Products of many conditional probabilities can become extremely small. Taking logarithms gives

$$
\log P(x_1,\ldots,x_T)
=\sum_{t=1}^{T}\log P(x_t\mid x_{<t}).
$$

The negative log-likelihood of an observed sequence is

$$
-\sum_{t=1}^{T}\log P(x_t\mid x_{<t}).
$$

Training encourages the model to assign larger conditional probability to the observed next token at each position.

This mathematical statement does not mean the model stores one fixed continuation. It represents a distribution over possible next tokens for each context.


In [ ]:
conditional_probabilities10 = np.array([0.20, 0.50, 0.40])
sequence_probability10 = np.prod(conditional_probabilities10)
sequence_log_probability10 = np.sum(np.log(conditional_probabilities10))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
axes[0].bar(["AI", "learns | AI", "quickly | context"], conditional_probabilities10)
axes[0].set_ylim(0, 1)
axes[0].set_ylabel("conditional probability")
axes[0].set_title("One factor per token")
partial_products10 = np.cumprod(conditional_probabilities10)
axes[1].plot([1, 2, 3], partial_products10, marker="o", linewidth=2)
axes[1].set_xticks([1, 2, 3])
axes[1].set_xlabel("number of included tokens")
axes[1].set_ylabel("partial sequence probability")
axes[1].set_title("Joint probability multiplies conditional factors")
plt.tight_layout()
plt.show()


## 5. Conditional generation versus causal understanding

A conditional language model estimates patterns of token continuation. The expression

$$
P(x_t\mid x_{<t})
$$

is probabilistic conditioning, not automatically a causal statement about why the text is true or why an event occurred.

Conditional probability can support useful prediction without identifying causal effects. Causal questions require additional assumptions, experimental designs, or causal models.


> **Common mistakes in Section 4**
>
> - Multiplying unconditional token probabilities instead of context-dependent probabilities.
> - Assuming the chain rule requires token independence.
> - Treating a token as always identical to a complete word.
> - Confusing probability of a complete sequence with the probability of one next token.
> - Interpreting statistical conditioning as proof of causation.

## Section 4 terminology

| Term | Meaning |
|---|---|
| Probability chain rule | Factorization of a joint probability into ordered conditionals |
| Token | Encoded text unit processed by a language model |
| Vocabulary | Set of tokens available to the model |
| Context | Earlier tokens used to condition the next-token distribution |
| Autoregressive model | Model generating or scoring one element conditioned on earlier elements |
| Sequence likelihood | Product of conditional probabilities for observed sequence elements |
| Decoding rule | Procedure selecting outputs from model probabilities |


## Section 4 guided worksheet

A model assigns

$$
P(x_1)=0.25,
$$

$$
P(x_2\mid x_1)=0.60,
$$

and

$$
P(x_3\mid x_1,x_2)=0.10.
$$

1. Compute the three-token sequence probability.
2. Compute its log-probability.
3. Write the negative log-likelihood.
4. Explain why the third factor cannot generally be replaced by $$P(x_3).$$


In [ ]:
reveal_markdown("Open Section 4 solutions", r"""
1. Multiply the conditional factors:

   $$
   P(x_1,x_2,x_3)=(0.25)(0.60)(0.10)
   =\boxed{0.015}.
   $$

2. The log-probability is

   $$
   \ln(0.25)+\ln(0.60)+\ln(0.10)
   =\ln(0.015)
   \approx \boxed{-4.199705}.
   $$

3. The negative log-likelihood is

   $$
   \boxed{-\ln(0.015)\approx 4.199705}.
   $$

4. The distribution of the third token can depend strongly on the earlier context. Replacing the conditional with its marginal would discard that information and would require an independence assumption not provided.
""")


---

# Section 5: Bayesian Inference and Sequential Updating

## 1. Hypotheses and evidence

In Bayesian inference, an unknown state or parameter is represented by competing hypotheses. A prior distribution expresses information before the current data, and Bayes' theorem produces a posterior distribution afterward.

For hypotheses $$H_1,\ldots,H_k$$ and evidence $$E,$$

$$
P(H_j\mid E)
=\frac{P(E\mid H_j)P(H_j)}
{\sum_iP(E\mid H_i)P(H_i)}.
$$

The posterior can become the prior for a later update.


## 2. Detailed model-reliability update

Suppose a deployed model is in one of two states:

$$
G=\text{well calibrated}
$$

or

$$
B=\text{poorly calibrated}.
$$

Before a validation check,

$$
P(G)=0.70,
\qquad
P(B)=0.30.
$$

Let $$T$$ mean that the model passes the check. Assume

$$
P(T\mid G)=0.90
$$

and

$$
P(T\mid B)=0.40.
$$

The evidence probability is

$$
P(T)=(0.90)(0.70)+(0.40)(0.30).
$$

Therefore

$$
P(T)=0.63+0.12=0.75.
$$

The posterior probability of good calibration is

$$
P(G\mid T)=\frac{(0.90)(0.70)}{0.75}
=\frac{0.63}{0.75}
=\boxed{0.84}.
$$


## 3. Sequential updating

After the first passing test, use

$$
P(G)=0.84
$$

as the new prior for a second conditionally independent test with the same likelihoods.

The unnormalized weights are

$$
P(T\mid G)P(G)=(0.90)(0.84)=0.756
$$

and

$$
P(T\mid B)P(B)=(0.40)(0.16)=0.064.
$$

Normalize:

$$
P(G\mid T,T)=\frac{0.756}{0.756+0.064}
=\frac{0.756}{0.82}
\approx \boxed{0.921951}.
$$

The conditional-independence assumption matters. If the tests share hidden errors, multiplying the same likelihood contributions can overstate the evidence.


In [ ]:
prior_good10 = 0.70
pass_given_good10 = 0.90
pass_given_bad10 = 0.40
posterior_sequence_good10 = [prior_good10]
for _ in range(5):
    current_prior = posterior_sequence_good10[-1]
    evidence = pass_given_good10*current_prior + pass_given_bad10*(1-current_prior)
    posterior_sequence_good10.append(pass_given_good10*current_prior/evidence)
posterior_sequence_good10 = np.array(posterior_sequence_good10)

fig, ax = plt.subplots()
ax.plot(np.arange(6), posterior_sequence_good10, marker="o", linewidth=2)
ax.set_ylim(0.65, 1.01)
ax.set_xticks(np.arange(6))
ax.set_xlabel("number of passing checks")
ax.set_ylabel("posterior probability of good calibration")
ax.set_title("Sequential evidence updates the posterior")
plt.show()


## 4. Bayesian prediction versus parameter estimation

Bayesian inference produces a distribution over unknown quantities rather than only one best estimate. Predictions can average over this uncertainty:

$$
P(Y_{\text{new}}\mid D)
=\int P(Y_{\text{new}}\mid\theta)P(\theta\mid D)\,d\theta.
$$

Here $$D$$ denotes observed data, $$\theta$$ is an unknown parameter, and $$P(\theta\mid D)$$ is the posterior. The integral is a weighted average of predictions across plausible parameter values.

This is conceptually different from plugging in a single estimate $$\widehat{\theta},$$ although the latter may be computationally simpler.


## 5. Priors are part of the model

A prior should reflect justified information, modeling assumptions, or a deliberately weak regularization choice. With limited data, it can substantially influence the posterior. With abundant informative data, the likelihood often has greater influence.

Bayesian conclusions are conditional on the prior, likelihood, and data-quality assumptions. Reporting sensitivity to reasonable prior choices can be important in high-stakes settings.


> **Common mistakes in Section 5**
>
> - Treating a prior as an observed-data frequency in every context.
> - Forgetting to normalize posterior weights.
> - Reusing dependent evidence as though observations were conditionally independent.
> - Confusing a posterior distribution with one point estimate.
> - Assuming a large dataset corrects a misspecified likelihood automatically.
> - Hiding strong prior assumptions in a high-stakes conclusion.

## Section 5 terminology

| Term | Meaning |
|---|---|
| Hypothesis | Candidate explanation or unknown state |
| Bayesian update | Transformation of prior into posterior using likelihood |
| Sequential updating | Using one posterior as the next prior |
| Posterior predictive distribution | Prediction averaged over posterior uncertainty |
| Parameter uncertainty | Uncertainty about an unknown model quantity |
| Model specification | Choice of likelihood, prior, and structural assumptions |
| Sensitivity analysis | Checking conclusions under reasonable assumption changes |


## Section 5 guided worksheet

Two models, $$M_1$$ and $$M_2,$$ have prior probabilities $$0.6$$ and $$0.4.$$ A validation pattern has probabilities

$$
P(E\mid M_1)=0.2
$$

and

$$
P(E\mid M_2)=0.5.
$$

1. Compute $$P(E).$$
2. Compute both posterior model probabilities.
3. Explain why evidence can favor the model with the smaller prior.
4. Verify that the posterior probabilities sum to one.


In [ ]:
reveal_markdown("Open Section 5 solutions", r"""
1. The evidence probability is

   $$
   P(E)=(0.2)(0.6)+(0.5)(0.4)
   =0.12+0.20
   =\boxed{0.32}.
   $$

2. For the first model,

   $$
   P(M_1\mid E)=\frac{(0.2)(0.6)}{0.32}
   =\frac{0.12}{0.32}
   =\boxed{0.375}.
   $$

   For the second,

   $$
   P(M_2\mid E)=\frac{(0.5)(0.4)}{0.32}
   =\frac{0.20}{0.32}
   =\boxed{0.625}.
   $$

3. Although $$M_2$$ begins with the smaller prior, the evidence is two and one-half times as likely under $$M_2$$ as under $$M_1,$$ which outweighs the prior difference.

4. We have

   $$
   0.375+0.625=1.
   $$
""")


---

# Section 6: Naive Bayes Text Classification

## 1. The classification rule

Let $$C$$ denote a document class and let $$x_1,\ldots,x_d$$ be observed features. Bayes' theorem gives

$$
P(C=c\mid x_1,\ldots,x_d)
\propto P(C=c)P(x_1,\ldots,x_d\mid C=c).
$$

Naive Bayes assumes features are conditionally independent given the class:

$$
P(x_1,\ldots,x_d\mid C=c)
=\prod_{j=1}^{d}P(x_j\mid C=c).
$$

Therefore the unnormalized class score is

$$
\boxed{S_c=P(C=c)\prod_{j=1}^{d}P(x_j\mid C=c).}
$$

Normalize scores if posterior probabilities are needed, or choose the class with largest score.


## 2. Bag-of-words representation

A **bag-of-words** representation records which words occur or how many times they occur while ignoring word order. The vocabulary supplies feature coordinates.

Different Naive Bayes variants use different feature models:

- Bernoulli Naive Bayes models word presence or absence;
- Multinomial Naive Bayes models token counts; and
- Gaussian Naive Bayes models continuous features with class-specific Gaussian distributions.

The word “naive” refers to the strong conditional-independence assumption, not to an absence of mathematical structure.


## 3. Detailed two-word classification

Suppose

$$
P(S)=0.4,
\qquad
P(H)=0.6.
$$

For the words “offer” and “project,” assume

$$
P(\text{offer}\mid S)=0.6,
\qquad
P(\text{project}\mid S)=0.1,
$$

and

$$
P(\text{offer}\mid H)=0.05,
\qquad
P(\text{project}\mid H)=0.4.
$$

For a document containing both words, the Naive Bayes spam score is

$$
S_S=(0.4)(0.6)(0.1)=0.024.
$$

The ham score is

$$
S_H=(0.6)(0.05)(0.4)=0.012.
$$

Normalize:

$$
P(S\mid\text{features})
=\frac{0.024}{0.024+0.012}
=\boxed{\frac{2}{3}}.
$$

The classifier chooses spam because its posterior score is larger.


## 4. Laplace smoothing

An unseen word-class pair can produce an estimated probability of zero. Multiplying by zero makes the entire class score zero.

**Laplace smoothing** adds a positive pseudocount, meaning an artificial count used for regularization rather than an additional observed token. For token counts,

$$
\boxed{P(\text{word }j\mid c)
=\frac{N_{jc}+\alpha}{N_c+\alpha V},}
$$

where:

- $$N_{jc}$$ is the word count in class $$c$$;
- $$N_c$$ is the total token count in class $$c$$;
- $$V$$ is vocabulary size; and
- $$\alpha>0$$ is the smoothing strength.

For add-one smoothing, $$\alpha=1.$$ Smoothing prevents zero probabilities but also changes every estimated probability.


## 5. Log scores

Products of many probabilities can **underflow**, meaning the floating-point representation rounds an extremely small positive product to zero. Because logarithm is increasing, maximizing a product score is equivalent to maximizing its log:

$$
\log S_c
=\log P(C=c)+\sum_{j=1}^{d}\log P(x_j\mid C=c).
$$

The classifier can compare log scores directly. To recover normalized probabilities stably, subtract the largest log score before exponentiating.


In [ ]:
prior_simple10 = np.array([0.4, 0.6])
likelihood_simple10 = np.array([[0.6, 0.1], [0.05, 0.4]])
scores_simple10 = prior_simple10*np.prod(likelihood_simple10, axis=1)
posterior_simple10 = scores_simple10/scores_simple10.sum()
print("Unnormalized scores [spam, ham]:", scores_simple10)
print("Posterior probabilities [spam, ham]:", posterior_simple10)


> **Common mistakes in Section 6**
>
> - Multiplying marginal feature probabilities instead of class-conditional probabilities.
> - Omitting class priors.
> - Treating conditional independence as literal word independence in ordinary language.
> - Allowing unseen words to force zero probability without considering smoothing.
> - Multiplying long probability lists instead of using log scores.
> - Mixing Bernoulli and Multinomial feature interpretations.

## Section 6 terminology

| Term | Meaning |
|---|---|
| Naive Bayes | Bayesian classifier with class-conditional feature independence |
| Bag of words | Text representation ignoring order while recording word features |
| Class prior | Probability of a class before document features |
| Class-conditional likelihood | Feature probability within a specified class |
| Laplace smoothing | Addition of pseudocounts to avoid zero estimates |
| Vocabulary size | Number of modeled word features |
| Log score | Sum-based logarithm of an unnormalized class score |
| Underflow | Numerical rounding of an extremely small positive value to zero |
| Multinomial Naive Bayes | Naive Bayes variant modeling token counts |


## Section 6 guided worksheet

Suppose

$$
P(S)=0.3,
\qquad
P(H)=0.7,
$$

and a document contains “win” and “meeting.” Assume

$$
P(\text{win}\mid S)=0.5,
\qquad
P(\text{meeting}\mid S)=0.1,
$$

$$
P(\text{win}\mid H)=0.02,
\qquad
P(\text{meeting}\mid H)=0.4.
$$

1. Compute both unnormalized scores.
2. Normalize them.
3. State the predicted class.
4. Identify where conditional independence was used.


In [ ]:
reveal_markdown("Open Section 6 solutions", r"""
The spam score is

$$
S_S=(0.3)(0.5)(0.1)=0.015.
$$

The ham score is

$$
S_H=(0.7)(0.02)(0.4)=0.0056.
$$

The normalizing sum is

$$
0.015+0.0056=0.0206.
$$

Therefore

$$
P(S\mid\text{features})
=\frac{0.015}{0.0206}
\approx \boxed{0.728155},
$$

and

$$
P(H\mid\text{features})
=\frac{0.0056}{0.0206}
\approx \boxed{0.271845}.
$$

The predicted class is spam. Conditional independence was used when replacing the joint word likelihood within each class by the product of the two individual word likelihoods.
""")


---

# Integrated Instructor Lab: Train a Smoothed Naive Bayes Classifier

Use two classes, spam $$S$$ and ham $$H,$$ with training-document counts

$$
N_S=4,
\qquad
N_H=6.
$$

The class priors are

$$
P(S)=\frac{4}{10}=0.4,
\qquad
P(H)=\frac{6}{10}=0.6.
$$

Use vocabulary

$$
\{\text{free},\text{meeting},\text{project},\text{win}\}
$$

with token counts:

| Class | free | meeting | project | win | Total |
|---|---:|---:|---:|---:|---:|
| Spam | 6 | 1 | 1 | 4 | 12 |
| Ham | 1 | 7 | 6 | 0 | 14 |

Use add-one smoothing with vocabulary size $$V=4.$$


## Compute smoothed probabilities

For spam, the denominator is

$$
N_S^{\text{tokens}}+V=12+4=16.
$$

Thus

$$
P(\text{free}\mid S)=\frac{6+1}{16}=\frac{7}{16},
$$

$$
P(\text{meeting}\mid S)=\frac{1+1}{16}=\frac{2}{16},
$$

$$
P(\text{project}\mid S)=\frac{1+1}{16}=\frac{2}{16},
$$

and

$$
P(\text{win}\mid S)=\frac{4+1}{16}=\frac{5}{16}.
$$

For ham, the denominator is

$$
14+4=18.
$$

Therefore

$$
P(\text{free}\mid H)=\frac{2}{18},
$$

$$
P(\text{meeting}\mid H)=\frac{8}{18},
$$

$$
P(\text{project}\mid H)=\frac{7}{18},
$$

and

$$
P(\text{win}\mid H)=\frac{1}{18}.
$$


## Classify “free win”

The spam score is

$$
S_S=(0.4)\left(\frac{7}{16}\right)\left(\frac{5}{16}\right)
=0.0546875.
$$

The ham score is

$$
S_H=(0.6)\left(\frac{2}{18}\right)\left(\frac{1}{18}\right)
\approx 0.00370370.
$$

Normalize:

$$
P(S\mid\text{free, win})
=\frac{0.0546875}{0.0546875+0.00370370}
\approx \boxed{0.936570}.
$$

The document is classified as spam.


In [ ]:
class_priors_lab10 = np.array([0.4, 0.6])
counts_lab10 = np.array([[6, 1, 1, 4], [1, 7, 6, 0]], dtype=float)
vocabulary_lab10 = ["free", "meeting", "project", "win"]
smoothed_likelihoods_lab10 = (counts_lab10+1)/(counts_lab10.sum(axis=1, keepdims=True)+4)

def naive_bayes_posterior_lab10(word_indices):
    scores = class_priors_lab10*np.prod(smoothed_likelihoods_lab10[:, word_indices], axis=1)
    return scores, scores/scores.sum()

scores_free_win_lab10, posterior_free_win_lab10 = naive_bayes_posterior_lab10([0, 3])
scores_meeting_project_lab10, posterior_meeting_project_lab10 = naive_bayes_posterior_lab10([1, 2])

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
axes[0].bar(["spam", "ham"], posterior_free_win_lab10, color=["crimson", "steelblue"])
axes[0].set_ylim(0, 1)
axes[0].set_ylabel("posterior probability")
axes[0].set_title("Document: free win")
axes[1].bar(["spam", "ham"], posterior_meeting_project_lab10,
            color=["crimson", "steelblue"])
axes[1].set_ylim(0, 1)
axes[1].set_ylabel("posterior probability")
axes[1].set_title("Document: meeting project")
plt.tight_layout()
plt.show()

print("Smoothed likelihoods by class:")
print(smoothed_likelihoods_lab10)
print("Posterior for free win:", posterior_free_win_lab10)
print("Posterior for meeting project:", posterior_meeting_project_lab10)


## Integrated lab tasks

1. Verify that each row of smoothed word probabilities sums to one.
2. Classify the document “meeting project” by hand.
3. Compute both classifications without smoothing and describe what changes.
4. Explain the role of the class prior.
5. Identify at least two limitations of this small model.


In [ ]:
reveal_markdown("Open integrated-lab solution", r"""
For spam,

$$
\frac{7+2+2+5}{16}=\frac{16}{16}=1.
$$

For ham,

$$
\frac{2+8+7+1}{18}=\frac{18}{18}=1.
$$

For “meeting project,” the spam score is

$$
S_S=(0.4)\left(\frac{2}{16}\right)\left(\frac{2}{16}\right)
=0.00625.
$$

The ham score is

$$
S_H=(0.6)\left(\frac{8}{18}\right)\left(\frac{7}{18}\right)
\approx 0.10370370.
$$

Thus

$$
P(S\mid\text{meeting, project})
=\frac{0.00625}{0.00625+0.10370370}
\approx \boxed{0.056842}.
$$

The document is classified as ham.

Without smoothing, the ham probability of “win” is zero, so “free win” receives zero ham score. Smoothing avoids an absolute conclusion based on one unseen training count.

The class prior favors ham before observing words because ham documents are more common in training. Limitations include tiny training data, ignored word order, conditional-independence assumptions, a four-word vocabulary, possible dataset shift, and no treatment of word meaning or context.
""")


---

# Module 10 Summary

1. Joint probabilities describe events together; marginals sum over unwanted variables.
2. Conditional probabilities change the reference population.
3. The law of total probability combines conditionals across a partition.
4. Bayes' theorem combines prior and likelihood, then normalizes by evidence.
5. Independence and conditional independence are different statements.
6. The sequence chain rule factors joint probabilities into ordered conditionals.
7. Generative language models predict next-token distributions conditional on context.
8. Bayesian inference updates distributions over unknown hypotheses or parameters.
9. Naive Bayes uses class-conditional feature factorization, smoothing, and often log scores.

## Cumulative terminology

| AI language | Probability interpretation |
|---|---|
| Feature-label table | Joint distribution estimate |
| Class posterior | Conditional probability of class given features |
| Base rate | Class prior |
| Next-token prediction | Conditional distribution given context |
| Sequence likelihood | Product of next-element conditional probabilities |
| Naive Bayes score | Prior times class-conditional feature likelihoods |
| Smoothing | Pseudocount regularization of probability estimates |

## Exit ticket

1. Why are $$P(A\mid B)$$ and $$P(B\mid A)$$ generally different?
2. What does the evidence denominator do in Bayes' theorem?
3. How can features be dependent overall but conditionally independent within class?
4. Why does a language-model sequence probability factor into conditionals?
5. Why is Laplace smoothing useful in text classification?
